# PneumoFusionNet — Phase 2: Multimodal Classification
## Image + Radiology Report (Indication / History) Fusion

### Project Overview
Phase 2 extends the image-only baseline (Phase 1) by incorporating **radiology report text** (Impression section).  
Both modalities are fused via concatenation and passed through a shared classifier.

The model combines:

- **Image Branch** → PneumoFusionNet (ResNet50 + DSC + GCSA) → 1024-d  
- **Text Branch**  → BioClinicalBERT [CLS] token → 768-d  
- **Fusion**       → Concatenate [1024 + 768] = 1792-d → Classifier

---

## Dataset

- MIMIC-CXR JPG (P10 folder — 139 labelled samples)
- Labels: NORMAL (No Finding = 1.0) / PNEUMONIA (Pneumonia = 1.0)
- Text: INDICATION / HISTORY section of each radiology report
- Split: **80% Train / 20% Test (stratified)**

---

## Notebook Structure

1. Environment Setup  
2. Dataset Loading  
3. 80/20 Train/Test Split  
4. Transforms & Tokeniser  
5. Dataset Class & DataLoaders  
6. Model Architecture (Image + Text + Fusion)  
7. Loss, Optimiser & Scheduler  
8. Training  
9. Evaluation & Metrics  
10. Save Features for Phase 3  

---

## Author
IIT Guwahati  
B.Sc. in Data Science & Artificial Intelligence  
📧 Email: ay346185@gmail.com

## 1. Environment Setup

In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, f1_score, auc
)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR        = os.path.abspath(os.path.join(os.getcwd(), '..'))
CSV_PATH        = os.path.join(BASE_DIR, 'dataset_139', 'mimic_multimodal_dataset_v2.csv')
PHASE1_WEIGHTS  = os.path.join(BASE_DIR, 'outputs', 'best_pneumofusion_mimic.pth')
SAVE_DIR        = os.path.join(BASE_DIR, 'outputs')
MODEL_PATH      = os.path.join(SAVE_DIR, 'best_phase2_multimodal.pth')
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Hyperparameters ────────────────────────────────────────────────────────
IMG_SIZE=224; BATCH_SIZE=8; NUM_EPOCHS=30; LR=1e-4; WEIGHT_DECAY=1e-4; NUM_CLASSES=2
MAX_TEXT_LEN=128; BERT_MODEL='emilyalsentzer/Bio_ClinicalBERT'
CLASSES=['NORMAL','PNEUMONIA']; MEAN=[0.485]; STD=[0.229]

print('BASE_DIR :', BASE_DIR)
print('CSV      :', CSV_PATH)
print('Exists   :', os.path.exists(CSV_PATH))
print('BERT     :', BERT_MODEL)

BASE_DIR : C:\2026\PneumoFusionNet\mimic\mimic_pilot_139
CSV      : C:\2026\PneumoFusionNet\mimic\mimic_pilot_139\dataset_139\mimic_multimodal_dataset_v2.csv
Exists   : True
BERT     : emilyalsentzer/Bio_ClinicalBERT


In [ ]:
df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print(df['label_name'].value_counts())

# ── Extract Indication/History from report (prevents target leakage) ──
import re
def extract_indication(report):
    if pd.isna(report): return 'no indication available'
    match = re.search(r'(INDICATION|HISTORY|REASON FOR EXAM|REASON FOR EXAMINATION):\s*(.*?)(?=\s*(?:IMPRESSION|FINDINGS|COMPARISON|TECHNIQUE|CONCLUSION|//)|\Z)', str(report), re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(2).strip()
    return 'no indication available'

df['indication'] = df['report'].apply(extract_indication)

# ── Balance classes: downsample Normal to match Pneumonia ──
pneumonia_df = df[df['label'] == 1]
normal_df    = df[df['label'] == 0]
n_pneumonia  = len(pneumonia_df)
print(f'Before balancing: Normal={len(normal_df)}, Pneumonia={n_pneumonia}')

normal_downsampled = normal_df.sample(n=n_pneumonia, random_state=SEED)
df = pd.concat([normal_downsampled, pneumonia_df]).reset_index(drop=True)
print(f'After balancing : Normal={(df.label==0).sum()}, Pneumonia={(df.label==1).sum()}, Total={len(df)}')

df.head(10)


In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5)
train_idx, test_idx = next(sgkf.split(df, df['label'], groups=df['subject_id']))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df  = df.iloc[test_idx].reset_index(drop=True)

print('Train:', len(train_df), '| Pneumonia:', train_df.label.sum(), '| Normal:', (train_df.label==0).sum())
print('Test :', len(test_df),  '| Pneumonia:', test_df.label.sum(),  '| Normal:', (test_df.label==0).sum())


In [6]:
tfms = {
    'train': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((256, 256)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.25, contrast=0.25),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
    'test': transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD)
    ]),
}
print('Image transforms ready.')

Image transforms ready.


In [7]:
print('Loading BioClinicalBERT tokeniser...')
tokeniser = AutoTokenizer.from_pretrained(BERT_MODEL)
print('Tokeniser loaded.')

# Quick test
sample_enc = tokeniser(
    'No acute cardiopulmonary process.',
    return_tensors='pt',
    max_length=MAX_TEXT_LEN,
    padding='max_length',
    truncation=True
)
print('Token IDs shape:', sample_enc['input_ids'].shape)

Loading BioClinicalBERT tokeniser...
Tokeniser loaded.
Token IDs shape: torch.Size([1, 128])


In [8]:
class MIMICMultimodalDataset(Dataset):
    """Returns (image_tensor, input_ids, attention_mask, label)"""
    def __init__(self, df, tokeniser, transform=None, max_len=MAX_TEXT_LEN):
        self.df        = df.reset_index(drop=True)
        self.tokeniser = tokeniser
        self.transform = transform
        self.max_len   = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ── Image ──────────────────────────────────────────────
        img = Image.open(row.image_path)
        if self.transform: img = self.transform(img)

        # ── Text (Indication) ──────────────────────────────────
        text = str(row.impression) if pd.notna(row.impression) else 'no indication available'
        enc  = self.tokeniser(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids      = enc['input_ids'].squeeze(0)
        attention_mask = enc['attention_mask'].squeeze(0)

        return img, input_ids, attention_mask, int(row.label)

print('MIMICMultimodalDataset defined.')

MIMICMultimodalDataset defined.


In [ ]:
# Fix image paths to new location
import os
OLD = r'C:\2026\project\mimic\MIMIC_CXR_JPG_P1'
NEW = r'C:\2026\PneumoFusionNet\mimic\mimic_pilot_139\mimic-cxr-jpg\2.1.0\files'

df['image_path'] = df['image_path'].str.replace(OLD, NEW, regex=False)
train_df['image_path'] = train_df['image_path'].str.replace(OLD, NEW, regex=False)
test_df['image_path'] = test_df['image_path'].str.replace(OLD, NEW, regex=False)

# Verify
ok  = df['image_path'].apply(os.path.exists).sum()
print(f'Images found: {ok} / {len(df)}')
print('Sample:', df.iloc[0]['image_path'])


In [ ]:
# Classes are balanced — use simple shuffle instead of WeightedRandomSampler
dataloaders = {
    'train': DataLoader(MIMICMultimodalDataset(train_df, tokeniser, tfms['train']),
                        batch_size=BATCH_SIZE, shuffle=True),
    'test':  DataLoader(MIMICMultimodalDataset(test_df,  tokeniser, tfms['test']),
                        batch_size=BATCH_SIZE, shuffle=False),
}
dataset_sizes = {'train': len(train_df), 'test': len(test_df)}
print('Dataset sizes:', dataset_sizes)

# Sanity check
imgs, ids, masks, lbls = next(iter(dataloaders['train']))
print('Image batch :', imgs.shape)
print('Input IDs   :', ids.shape)
print('Attn masks  :', masks.shape)
print('Labels      :', lbls)


In [12]:
class GCSA(nn.Module):
    """Global Channel-Spatial Attention"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool     = nn.AdaptiveAvgPool2d(1)
        self.max_pool     = nn.AdaptiveMaxPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid      = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        B, C, H, W = x.shape
        avg    = self.avg_pool(x).view(B, C)
        mx     = self.max_pool(x).view(B, C)
        ch_att = self.sigmoid(self.mlp(avg) + self.mlp(mx)).view(B, C, 1, 1)
        x      = x * ch_att
        sp     = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], dim=1)
        return x * self.sigmoid(self.conv_spatial(sp))


class DSC(nn.Module):
    """Depthwise Separable Convolution"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch,  3, padding=1, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)

print('GCSA and DSC defined.')

GCSA and DSC defined.


In [13]:
class ImageEncoder(nn.Module):
    """ResNet50 + DSC + GCSA → 1024-d image features"""
    def __init__(self, freeze_until=6):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # 3-channel → 1-channel grayscale
        resnet.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            resnet.conv1.weight = nn.Parameter(
                resnet.conv1.weight.mean(dim=1, keepdim=True)
            )
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])
        for i, child in enumerate(self.backbone.children()):
            if i < freeze_until:
                for p in child.parameters(): p.requires_grad = False
        self.dsc  = DSC(2048, 1024)
        self.gcsa = GCSA(1024)
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.backbone(x)           # [B, 2048, 7, 7]
        x = self.dsc(x)                # [B, 1024, 7, 7]
        x = self.gcsa(x)               # [B, 1024, 7, 7]
        return self.pool(x).flatten(1) # [B, 1024]

print('ImageEncoder defined.')


ImageEncoder defined.


In [15]:
class TextEncoder(nn.Module):
    """BioClinicalBERT [CLS] token → 768-d text features"""
    def __init__(self, model_name=BERT_MODEL, freeze_layers=10):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        # Freeze first 10 layers — only fine-tune last 2
        for i, layer in enumerate(self.bert.encoder.layer):
            if i < freeze_layers:
                for p in layer.parameters(): p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # [B, 768] — [CLS] token

print('TextEncoder defined.')

TextEncoder defined.


In [16]:
class PneumoFusionNetV2(nn.Module):
    """Image (1024-d) + Text (768-d) → Fusion → Classification"""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.image_encoder = ImageEncoder()
        self.text_encoder  = TextEncoder()
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1024 + 768, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, imgs, input_ids, attention_mask):
        img_feat  = self.image_encoder(imgs)                     # [B, 1024]
        text_feat = self.text_encoder(input_ids, attention_mask) # [B, 768]
        fused     = torch.cat([img_feat, text_feat], dim=1)      # [B, 1792]
        return self.classifier(fused)                            # [B, 2]

    def get_features(self, imgs, input_ids, attention_mask):
        """1792-d fused embedding for Phase 3."""
        img_feat  = self.image_encoder(imgs)
        text_feat = self.text_encoder(input_ids, attention_mask)
        return torch.cat([img_feat, text_feat], dim=1)

print('PneumoFusionNetV2 defined.')

PneumoFusionNetV2 defined.


In [17]:
model = PneumoFusionNetV2().to(DEVICE)

# Test with dummy inputs
dummy_img  = torch.randn(2, 1, 224, 224).to(DEVICE)
dummy_ids  = torch.randint(0, 1000, (2, MAX_TEXT_LEN)).to(DEVICE)
dummy_mask = torch.ones(2, MAX_TEXT_LEN, dtype=torch.long).to(DEVICE)

print('Output shape  :', model(dummy_img, dummy_ids, dummy_mask).shape)
print('Feature shape :', model.get_features(dummy_img, dummy_ids, dummy_mask).shape)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,} ({trainable/total*100:.1f}%)')

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Output shape  : torch.Size([2, 2])
Feature shape : torch.Size([2, 1792])
Total params    : 134,980,965
Trainable params: 62,663,589 (46.4%)


In [ ]:
# Classes are balanced — no class weights needed
criterion     = nn.CrossEntropyLoss()
optimizer     = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler     = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print(f'Loss         : CrossEntropyLoss (balanced classes, no weights needed)')
print(f'Optimizer    : AdamW  (lr={LR})')
print(f'Scheduler    : CosineAnnealingLR (T_max={NUM_EPOCHS})')


In [ ]:
train_losses = []
best_loss    = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0; correct = 0; n = 0

    for imgs, input_ids, attention_mask, labels in dataloaders['train']:
        imgs           = imgs.to(DEVICE)
        input_ids      = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        labels         = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs, input_ids, attention_mask)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(labels)
        correct    += (outputs.argmax(1) == labels).sum().item()
        n          += len(labels)

    scheduler.step()
    epoch_loss = total_loss / n
    epoch_acc  = correct / n
    train_losses.append(epoch_loss)

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), MODEL_PATH)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.3f}')

print(f'\nBest Train Loss : {best_loss:.4f}')
print(f'Model saved to  : {MODEL_PATH}')

## 9. Training Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, NUM_EPOCHS+1), train_losses, color='royalblue', lw=2)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Phase 2 — Multimodal Training Loss Curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_train_loss.png'), dpi=120)
plt.show()


## 10. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for imgs, input_ids, attention_mask, labels in dataloaders['test']:
        imgs = imgs.to(DEVICE)
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)
        
        outputs = model(imgs, input_ids, attention_mask)
        probs = F.softmax(outputs, dim=1)[:, 1]
        preds = outputs.argmax(dim=1)
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print('--- Classification Report (Test Set) ---')
print(classification_report(all_labels, all_preds, target_names=CLASSES))


## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_confusion_matrix.png'), dpi=120)
plt.show()


## 12. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc     = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Test Set')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_roc_curve.png'), dpi=120)
plt.show()
print(f'AUC: {roc_auc:.4f}')


## 13. Final Metrics Summary

In [ ]:
acc = accuracy_score(all_labels, all_preds)
f1  = f1_score(all_labels, all_preds, zero_division=0)
print('=' * 40)
print('  PHASE 2 — TEST SET RESULTS')
print('=' * 40)
print(f'  Accuracy  : {acc:.4f}')
print(f'  F1 Score  : {f1:.4f}')
print(f'  AUC       : {roc_auc:.4f}')
print('=' * 40)
print()
print('Outputs saved to:', SAVE_DIR)
print('  - best_phase2_multimodal.pth')
print('  - phase2_train_loss.png')
print('  - phase2_confusion_matrix.png')
print('  - phase2_roc_curve.png')


## 14. Save Features for Phase 3

In [ ]:
def extract_features(dataloader, desc='Extracting'):
    features = []
    lbls = []
    with torch.no_grad():
        for imgs, input_ids, attention_mask, labels in dataloader:
            imgs = imgs.to(DEVICE)
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)
            feat = model.get_features(imgs, input_ids, attention_mask)
            features.append(feat.cpu())
            lbls.append(labels)
    return torch.cat(features), torch.cat(lbls)

print('Extracting Train Features...')
train_feats, train_labels_out = extract_features(dataloaders['train'])
print('Extracting Test Features...')
test_feats, test_labels_out = extract_features(dataloaders['test'])

train_feat_path = os.path.join(SAVE_DIR, 'train_multimodal_features.pt')
test_feat_path = os.path.join(SAVE_DIR, 'test_multimodal_features.pt')

torch.save({'features': train_feats, 'labels': train_labels_out}, train_feat_path)
torch.save({'features': test_feats, 'labels': test_labels_out}, test_feat_path)

print(f'Saved Train Features: {train_feat_path} {train_feats.shape}')
print(f'Saved Test Features: {test_feat_path} {test_feats.shape}')
